In [24]:
import pandas as pd
import numpy as np

In [25]:
listOfOrders = pd.read_excel("../data/raw/List_of_Orders.xlsx")
orderDetails = pd.read_excel("../data/raw/Order_Details.xlsx")
salesTarget = pd.read_excel("../data/raw/Sales_target.xlsx")

In [26]:
orders = listOfOrders.copy()
details = orderDetails.copy()
targets = salesTarget.copy()

In [27]:
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.strip()                # Remove leading/trailing spaces
        .str.lower()                # Convert to lowercase
        .str.replace(" ", "_")      # Replace spaces with _
        .str.replace("-", "_")      # Replace hyphens with _
    )
    return df

orders = clean_column_names(orders)
details = clean_column_names(details)
targets = clean_column_names(targets)

In [28]:
print(orders.columns)
print(details.columns)
print(targets.columns)

Index(['order_id', 'order_date', 'customername', 'state', 'city'], dtype='str')
Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category'], dtype='str')
Index(['month_of_order_date', 'category', 'target'], dtype='str')


In [29]:
def clean_text_columns(df):
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for col in text_columns:
        if df[col].map(lambda x: isinstance(x, str)).all():
            df[col] = df[col].str.strip()

    return df

orders = clean_text_columns(orders)
details = clean_text_columns(details)
targets = clean_text_columns(targets)

In [30]:
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [31]:
orders.info()

details.info()

targets.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      500 non-null    str           
 1   order_date    500 non-null    datetime64[us]
 2   customername  500 non-null    str           
 3   state         500 non-null    str           
 4   city          500 non-null    str           
dtypes: datetime64[us](1), str(4)
memory usage: 19.7 KB
<class 'pandas.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   order_id      1522 non-null   str  
 1   amount        1522 non-null   int64
 2   profit        1522 non-null   int64
 3   quantity      1522 non-null   int64
 4   category      1522 non-null   str  
 5   sub_category  1522 non-null   str  
dtypes: int64(3), str(3)
memory usage: 71.5 KB
<class 'pandas.DataFrame'>
Rang

In [32]:
invalid_order_ids = details.loc[
    ~details["order_id"].isin(orders["order_id"])
]

print(f"Invalid Order IDs: {len(invalid_order_ids)}")

invalid_order_ids.head()

Invalid Order IDs: 0


,order_id,amount,profit,quantity,category,sub_category


In [33]:
print("Orders Table:", orders["order_id"].nunique())
print("Details Table:", details["order_id"].nunique())

Orders Table: 500
Details Table: 500


In [34]:
orders.to_csv(
    "../data/processed/orders_clean.csv",
    index=False
)

details.to_csv(
    "../data/processed/order_details_clean.csv",
    index=False
)

targets.to_csv(
    "../data/processed/sales_target_clean.csv",
    index=False
)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


In [35]:
print(targets.columns)
print(targets.head())
print(targets.isnull().sum())

Index(['month_of_order_date', 'category', 'target'], dtype='str')
   month_of_order_date   category   target
0  2026-04-19 00:00:00  Furniture  11900.0
1  2026-05-19 00:00:00  Furniture  12000.0
2  2026-06-19 00:00:00  Furniture  12100.0
3  2026-07-19 00:00:00  Furniture  12300.0
4  2026-08-19 00:00:00  Furniture  12400.0
month_of_order_date    1
category               2
target                 2
dtype: int64


In [36]:
# Remove completely empty rows
targets = targets.dropna(how="all")

# Remove note/footer rows
targets = targets[
    ~targets["month_of_order_date"]
    .astype(str)
    .str.startswith("Note:")
]

# Remove any remaining rows with missing required values
targets = targets.dropna(
    subset=["month_of_order_date", "category", "target"]
)

In [37]:
print(targets.tail())
print(targets.isnull().sum())

    month_of_order_date     category   target
13  2026-05-19 00:00:00  Electronics  16000.0
14  2026-06-19 00:00:00  Electronics  16000.0
15  2026-07-19 00:00:00  Electronics  16000.0
16  2026-08-19 00:00:00  Electronics  16000.0
17  2026-09-19 00:00:00  Electronics  16000.0
month_of_order_date    0
category               0
target                 0
dtype: int64
